# Lovely Professional University
## Course: INT395 - Supervised Learning
### Capstone Project: LPU Student Academic Early-Warning & Intervention System
---
**Notebook**: Exploratory Data Analysis, Leakage-Free Preprocessing Pipeline, Multi-Model Benchmarking, Hyperparameter Tuning, Explainable AI (SHAP), and Actionable Intervention Mapping.

**Author**: Senior ML Engineer & Full-Stack Developer  
**Architecture**: Scikit-Learn ColumnTransformer + imblearn Pipeline + SMOTE + XGBoost + TreeExplainer SHAP

## 1. Environment Setup & Library Imports
We load the standard scientific Python stack alongside specialized machine learning and interpretability libraries.

In [1]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn & Imbalanced-learn
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
import shap

# Ensure local modules can be imported
sys.path.append(os.path.abspath('..'))
from src.custom_transformers import AcademicFeatureEngineer, IQROutlierCapper
from src.explain import explain_student_risk, generate_actionable_interventions

# Styling
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['figure.dpi'] = 120
SEED = 42
np.random.seed(SEED)
print('Libraries successfully imported and environment configured.')

## 2. Dataset Acquisition & Schema Exploration
We load `data/student_academic_data.csv` containing 2,500 LPU student records with simulated 3% missingness and realistic academic distributions.

In [2]:
data_path = os.path.join('..', 'data', 'student_academic_data.csv')
if not os.path.exists(data_path):
    data_path = os.path.join('data', 'student_academic_data.csv')
df = pd.read_csv(data_path)
print(f'Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(5)

In [3]:
# Summary statistics and missing value check
print('--- Missing Values Count ---')
print(df.isna().sum()[df.isna().sum() > 0])
print('\n--- Target Class Distribution ---')
print(df['Risk_Level'].value_counts(normalize=True).round(3) * 100)

## 3. Exploratory Data Analysis (EDA)
We inspect key feature correlations, attendance thresholds (LPU 75% cutoff), and academic performance drivers.

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Target Risk Distribution
risk_palette = {'Low': '#16A34A', 'Medium': '#D97706', 'High': '#DC2626'}
sns.countplot(data=df, x='Risk_Level', order=['Low', 'Medium', 'High'], palette=risk_palette, ax=axes[0], edgecolor='#0F172A')
axes[0].set_title('Target Class Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Academic Risk Category')
axes[0].set_ylabel('Number of Students')

# 2. Attendance % vs MidTerm Marks colored by Risk
sns.scatterplot(data=df, x='Attendance_Pct', y='MidTerm_Marks', hue='Risk_Level', palette=risk_palette, alpha=0.75, ax=axes[1])
axes[1].axvline(75.0, color='#0F172A', linestyle='--', label='75% Mandatory Cutoff')
axes[1].set_title('Attendance % vs. Mid-Term Marks', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Attendance (%)')
axes[1].set_ylabel('Mid-Term Marks (0-100)')
axes[1].legend(title='Risk Level')

plt.tight_layout()
plt.show()

In [5]:
# Correlation Heatmap across numeric indicators
num_cols = df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 8))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', cbar=True, square=True, linewidths=0.5)
plt.title('Academic Telemetry Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Leakage-Free Stratified Data Splitting
To guarantee absolute protection against data leakage, we divide data into **Train (70%)**, **Validation (15%)**, and **Test (15%)** sets using stratified sampling before any feature imputation, scaling, or transformation.

In [6]:
target_map = {'Low': 0, 'Medium': 1, 'High': 2}
X = df.drop(columns=['Student_ID', 'Risk_Level'])
y = df['Risk_Level'].map(target_map)

# 1st Split: 85% Train+Val, 15% Held-out Test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=SEED
)

# 2nd Split: 70% Train, 15% Validation (15/85 = ~0.17647)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=(0.15 / 0.85), stratify=y_train_val, random_state=SEED
)

print(f'Training set  : {X_train.shape[0]} records ({X_train.shape[0]/len(df):.1%})')
print(f'Validation set: {X_val.shape[0]} records ({X_val.shape[0]/len(df):.1%})')
print(f'Held-out Test : {X_test.shape[0]} records ({X_test.shape[0]/len(df):.1%})')

## 5. Feature Engineering & ColumnTransformer Pipeline
We encapsulate feature engineering, imputation, IQR outlier clipping, one-hot encoding, standard scaling, and SMOTE within an imblearn Pipeline.

In [7]:
from src.train import build_preprocessing_pipeline

fe, preprocessor, num_features, cat_features = build_preprocessing_pipeline()
print('Preprocessing pipeline configured with:')
print(f' - Numeric Features ({len(num_features)}): {num_features[:5]}...')
print(f' - Categorical Features ({len(cat_features)}): {cat_features}')

## 6. Multi-Model Benchmark & High-Risk Recall Prioritization
We benchmark 5 distinct classifiers starting from Logistic Regression (Baseline) through Ensembles (Random Forest, XGBoost, LightGBM) and Support Vector Machines.

In [8]:
# Load the trained benchmark comparison summary from artifacts
comp_path = os.path.join('..', 'artifacts', 'model_comparison.json')
if not os.path.exists(comp_path):
    comp_path = os.path.join('artifacts', 'model_comparison.json')
with open(comp_path, 'r') as f:
    comp_data = json.load(f)

comp_df = pd.DataFrame(comp_data).T
display_cols = ['Accuracy', 'High_Risk_Recall', 'High_Risk_Precision', 'F1_Macro', 'ROC_AUC_Macro']
comp_df[display_cols]

## 7. Model Diagnostics: Confusion Matrix & ROC Curves
We inspect classification performance on the test set for the winning Tuned XGBoost model.

In [9]:
# Load Confusion Matrix and ROC Curve Data
cm_path = os.path.join('..', 'artifacts', 'confusion_matrix.json')
if not os.path.exists(cm_path):
    cm_path = os.path.join('artifacts', 'confusion_matrix.json')
with open(cm_path) as f:
    cm_data = json.load(f)

cm = np.array(cm_data['matrix_raw'])
labels = cm_data['labels']

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title('Test Set Confusion Matrix (Tuned XGBoost)', fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted Risk Level')
ax.set_ylabel('Actual Risk Level')
plt.tight_layout()
plt.show()

## 8. Explainable AI (XAI) with SHAP & Actionable Interventions
We compute local SHAP feature attributions and map them to concrete institutional intervention protocols.

In [10]:
pipe_path = os.path.join('..', 'artifacts', 'best_model_pipeline.joblib')
if not os.path.exists(pipe_path):
    pipe_path = os.path.join('artifacts', 'best_model_pipeline.joblib')
shap_path = os.path.join('..', 'artifacts', 'shap_explainer.joblib')
if not os.path.exists(shap_path):
    shap_path = os.path.join('artifacts', 'shap_explainer.joblib')
fn_path = os.path.join('..', 'artifacts', 'feature_names.json')
if not os.path.exists(fn_path):
    fn_path = os.path.join('artifacts', 'feature_names.json')

pipeline = joblib.load(pipe_path)
explainer = joblib.load(shap_path)
with open(fn_path) as f:
    feature_names = json.load(f)

# Select a test student
sample_student = X_test.iloc[[0]]
shap_df, probs, pred_label = explain_student_risk(sample_student, pipeline, explainer, feature_names)
print(f'Predicted Risk: {pred_label} | Probabilities: Low={probs[0]:.2f}, Medium={probs[1]:.2f}, High={probs[2]:.2f}')
print('\nTop Local SHAP Risk Drivers:')
print(shap_df.head(5)[['Display_Feature', 'SHAP_Value', 'Student_Value']])

# Generate Actionable Mentor Interventions
interventions = generate_actionable_interventions(sample_student.iloc[0].to_dict())
print(f'\nTriggered Actionable Mentor Interventions ({len(interventions)}):')
for idx, action in enumerate(interventions, 1):
    print(f"{idx}. [{action['priority']}] {action['title']}: {action['action']}")

## 9. Academic Defense & Viva Key Takeaways
1. **Data Leakage Prevention**: Stratified splitting occurred strictly before imputers, capper bounds, or scalers were fit.
2. **Evaluation Metric Priority**: High-Risk Recall is optimized to prevent fatal False Negatives where struggling students are left unassisted.
3. **Explainable AI Integration**: SHAP bridges the gap between black-box tree models and pedagogical action plans for university mentors.
4. **Institutional Governance**: Human-in-the-loop guarantee ensures ethical decision support without automated adverse actions.